In [3]:
### Install Requireme
#!pip install chromadb==0.5.5 langchain-chroma==0.1.2 langchain==0.2.11 langchain-community==0.2.10 langchain-text-splitters==0.2.2 langchain-groq==0.1.6 transformers==4.43.2 sentence-transformers==3.0.1 unstructured==0.15.0 unstructured[pdf]==0.15.0
# Install poppler-utils
#!apt-get install poppler-utils -y
#!apt-get install tesseract-ocr -y
#!apt-get install libtesseract-dev -y

In [1]:
import os
import nltk
import ssl
import zipfile

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

# Download the punkt_tab data if it doesn't exist
nltk_data_path = nltk.data.find(".")
punkt_tab_path = os.path.join(nltk_data_path, "tokenizers", "punkt_tab.zip")
# Correctly specify the extracted directory 'punkt_tab'
punkt_tab_dir = os.path.join(nltk_data_path, "tokenizers", "punkt_tab")

if not os.path.exists(punkt_tab_dir): # Check if the extracted directory exists
    if not os.path.exists(punkt_tab_path):
        import urllib.request
        url = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages/tokenizers/punkt_tab.zip"
        # Download the data to the destination
        with urllib.request.urlopen(url) as response, open(punkt_tab_path, 'wb') as out_file:
            data = response.read()
            out_file.write(data)
        print(f"Downloaded punkt_tab.zip to {punkt_tab_path}")
    else:
        print(f"punkt_tab.zip already exists at {punkt_tab_path}")

    # Extract the contents of the zip file to the correct location
    with zipfile.ZipFile(punkt_tab_path, 'r') as zip_ref:
        # Extract to the parent directory 'tokenizers'
        zip_ref.extractall(os.path.dirname(punkt_tab_dir))
    print(f"Extracted punkt_tab.zip to {os.path.dirname(punkt_tab_dir)}")
else:
    print(f"punkt_tab directory already exists at {punkt_tab_dir}")

# The rest of your import statements...
from langchain.document_loaders import UnstructuredFileLoader
# ...
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA

punkt_tab directory already exists at C:\Users\Ahmed Ashraf\AppData\Roaming\nltk_data\.\tokenizers\punkt_tab


In [2]:
os.environ["GROQ_API_KEY"] = "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"

In [ ]:
# laoding the document
loader = UnstructuredFileLoader("notebookd8e32598e0 (1)pdf")
documents = loader.load()

In [6]:
# splitting into text chunks
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=400
)
texts = text_splitter.split_documents(documents)
type(texts),len(texts)

In [10]:
embeddings = HuggingFaceEmbeddings()

C:\Users\Ahmed Ashraf\AppData\Local\Temp\ipykernel_876\3655315981.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings()


In [11]:
persist_directory = "doc_db"

In [12]:
vectordb = Chroma.from_documents(
    documents=texts,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [13]:
# retriever
retriever = vectordb.as_retriever()

In [15]:
# llm from groq
llm = ChatGroq(
    model="llama-3.1-70b-versatile",
    temperature=0
)

In [16]:
# create a qa chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [65]:
######### ASK
def ASK(query):
    response = qa_chain.invoke({"query": query})
    print(query)
    print("\n","*"*80,"\n")
    print(response["result"])
    print("\n","*"*80,"\n")
    print("Source Document:", response["source_documents"][0].metadata["source"])

In [66]:
# invoke the qa chain and get a response for user query
query = "What about this Document?"
ASK(query)

What about this Document?

 ******************************************************************************** 

This document appears to be a set of lecture slides for a data mining or machine learning course, specifically covering the topic of clustering. The slides are from a course called CS583, taught by Bing Liu at the University of Illinois at Chicago (UIC).

The slides cover various aspects of clustering, including:

1. Basic concepts: types of attributes (symmetric, asymmetric, nominal), distance functions, and clustering algorithms (K-means, hierarchical).
2. Clustering evaluation: methods for evaluating the quality of a clustering result, including external evaluation (using labeled data) and internal evaluation (using measures such as intra-cluster cohesion and inter-cluster separation).
3. Clustering applications: examples of how clustering is used in various fields, such as marketing, text document organization, and recommendation systems.
4. Clustering algorithms: K-means 

In [67]:
# invoke the qa chain and get a response for user query
query = "Can you provide summary for kmeans in this document?"
ASK(query)

Can you provide summary for kmeans in this document?

 ******************************************************************************** 

Based on the provided context, here is a summary of K-means clustering:

**K-means Clustering**

* K-means is a partitional clustering algorithm.
* It partitions the given data into k clusters, where k is specified by the user.
* Each cluster has a cluster center, called a centroid.
* The algorithm works as follows:
  1. Randomly choose k data points (seeds) to be the initial centroids.
  2. Assign each data point to the closest centroid.
  3. Re-compute the centroids using the current cluster memberships.
  4. Repeat steps 2-3 until a convergence criterion is met.

**Convergence Criterion**

* No (or minimum) re-assignments of data points to different clusters.
* No (or minimum) change of centroids.
* Minimum decrease in the sum of squared error (SSE).

**Strengths**

* Not explicitly mentioned in the provided context, but mentioned that there is a 

In [68]:
# invoke the qa chain and get a response for user query
query = "provide summary for all document?"
ASK(query)

provide summary for all document?

 ******************************************************************************** 

The document appears to be a lecture on clustering, a data mining technique used to group similar data points or objects into clusters. Here's a summary of the key points:

**What is Clustering?**

* Clustering is a technique used to group similar data points or objects into clusters.
* It is used in various fields such as medicine, psychology, botany, sociology, biology, archeology, marketing, insurance, and libraries.
* Clustering is used to identify patterns or structures in data that are not easily visible.

**Examples of Clustering**

* Grouping people of similar sizes together to make "small", "medium", and "large" T-Shirts.
* Segmenting customers in marketing according to their similarities.
* Organizing text documents according to their content similarities.

**Aspects of Clustering**

* Clustering algorithm: Partitional clustering, Hierarchical clustering, etc

In [69]:
# invoke the qa chain and get a response for user query
query = "provide summary for clustering types?"
ASK(query)

provide summary for clustering types?

 ******************************************************************************** 

Based on the provided context, here is a summary of clustering types:

1. **Partitional Clustering**: Divides the data into a fixed number of clusters, where each cluster is a separate group of data points.
2. **Hierarchical Clustering**: Produces a nested sequence of clusters, a tree, also called a Dendrogram. It can be further divided into:
	* **Agglomerative (Bottom-up) Clustering**: Builds the dendrogram from the bottom level, merging the most similar pair of clusters until all data points are merged into a single cluster.
	* **Divisive (Top-down) Clustering**: Not explicitly mentioned in the provided context, but it starts with all data points in a single cluster and recursively splits the cluster into smaller ones.

Note that the provided context does not mention other types of clustering, such as **Density-based Clustering**, **Distribution-based Clustering*

In [70]:
# invoke the qa chain and get a response for user query
query = "provide summary for Hierarchical Clustering and how to apply?"
ASK(query)

provide summary for Hierarchical Clustering and how to apply?

 ******************************************************************************** 

**Hierarchical Clustering:**

Hierarchical clustering is a type of clustering algorithm that produces a nested sequence of clusters, represented as a tree-like structure called a dendrogram. This algorithm starts with each data point as a separate cluster and then merges the most similar clusters until all data points are merged into a single cluster.

**Types of Hierarchical Clustering:**

1. **Agglomerative (Bottom-up) Clustering:** This approach starts with each data point as a separate cluster and then merges the most similar clusters until all data points are merged into a single cluster.
2. **Divisive (Top-down) Clustering:** This approach starts with all data points in a single cluster and then splits the cluster into smaller clusters until each data point is in its own cluster.

**How to Apply Hierarchical Clustering:**

1. **Choose 